# Beginner: Run Easy SeqTrainer BenchLab Models From JSON

This notebook shows the easiest path for running the lightweight local models from a saved `run_config.json`.

It runs:

- Linear Regression
- Random Forest
- Gradient Boosting

The important idea: **the model settings come from JSON**, not from hidden notebook variables.

## Step 0: What This Notebook Uses

This notebook uses a tiny example dataset and config already included in the repo:

- Dataset: `examples/reproducibility/easy_promoters.csv`
- Config: `examples/reproducibility/easy_run_config.json`

Later, you can replace the config path with a `run_config.json` exported by BenchLab.

In [ ]:
from pathlib import Path
import sys

# Make imports work whether this notebook is opened from the repo root or from notebooks/.
repo_root = Path.cwd()
if not (repo_root / "app").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repo root:", repo_root)

## Step 1: Load The JSON Config

The config stores the dataset path, checksum, preprocessing choices, model list, split seed, and training settings.

In [ ]:
from app.reproducibility.config import ReproducibleRunConfig

config_path = repo_root / "examples" / "reproducibility" / "easy_run_config.json"
config = ReproducibleRunConfig.load(config_path)

print("Schema version:", config.schema_version)
print("Dataset:", config.dataset.path)
print("Sequence column:", config.dataset.sequence_column)
print("Target column:", config.dataset.target_column)
print("Models:", [model.model_name for model in config.models])
print("Random seed:", config.split.random_seed)

## Step 2: Check The Dataset

BenchLab stores a SHA256 checksum in the JSON. If the local dataset exists, we can confirm it matches the saved config.

In [ ]:
from app.reproducibility.run_from_config import verify_dataset_checksum

print("Checksum verified:", verify_dataset_checksum(config))

## Step 3: Preview The Dataset

This is just to help you see what the example data looks like.

In [ ]:
import pandas as pd

dataset_path = repo_root / config.dataset.path
df = pd.read_csv(dataset_path)
df.head()

## Step 4: Dry Run First

A dry run validates the config and writes reproducibility files, but it does not train models. This is a safe first check.

In [ ]:
from app.reproducibility.run_from_config import replay_from_config

dry_output = repo_root / "storage" / "notebook_beginner_dry_run"
dry_summary = replay_from_config(config_path, output_dir=dry_output, dry_run=True)
dry_summary

## Step 5: Train The Easy Models

Now we run the supported local models from the same JSON config.

This creates metrics, predictions, model files, and reproducibility artifacts.

In [ ]:
run_output = repo_root / "storage" / "notebook_beginner_easy_run"
run_summary = replay_from_config(config_path, output_dir=run_output, dry_run=False)
run_summary

## Step 6: Read The Metrics

The output metrics are saved as JSON. For binary labels, BenchLab also reports thresholded metrics such as accuracy, precision, recall, F1, and MCC.

In [ ]:
import json

metrics_path = Path(run_summary["metrics_path"])
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
metrics

## Step 7: Read The Predictions

Predictions are saved as CSV so they can be compared later with CNN, DNABERT2, or iPro-MP outputs.

In [ ]:
predictions_path = Path(run_summary["predictions_path"])
predictions = pd.read_csv(predictions_path)
predictions.head()

## Step 8: What Files Were Created?

These files help reproduce or audit the run later.

In [ ]:
for path in sorted(run_output.iterdir()):
    print(path.name)

## Step 9: Use Your Own BenchLab Export

After running BenchLab in the web app, download/export a run bundle and find its `run_config.json`.

Then replace this line:

```python
config_path = repo_root / "examples" / "reproducibility" / "easy_run_config.json"
```

with your own config path, for example:

```python
config_path = Path("C:/path/to/my/run_config.json")
```

Important: if you want to train from the JSON, the dataset path in the config must point to a dataset file that exists on your machine.